# LoRA Fine-Tuning with PEFT

Fine-tune a causal language model on a custom Q&A dataset using Low-Rank Adaptation (LoRA).
LoRA inserts trainable rank-decomposition matrices into frozen transformer weights,
cutting trainable parameters by ~99% vs full fine-tuning while matching most of its quality.

**What this notebook covers:**
- Dataset preparation from JSONL (instruction / response pairs)
- LoRA configuration with PEFT
- Supervised fine-tuning loop
- Saving and loading the LoRA adapter separately from the base model
- Running inference with the merged weights

## 1. Install dependencies

In [ ]:
# !pip install -q transformers>=4.40.0 peft>=0.10.0 datasets accelerate bitsandbytes trl

In [ ]:
import json
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 2. Configuration

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
BASE_MODEL = "distilgpt2"           # swap for "meta-llama/Llama-2-7b-hf" etc.
OUTPUT_DIR = "./lora_adapter"
MERGED_DIR = "./lora_merged"

# ── LoRA ──────────────────────────────────────────────────────────────────────
LORA_R = 8                          # rank of the update matrices
LORA_ALPHA = 32                     # LoRA scaling factor
LORA_DROPOUT = 0.05
TARGET_MODULES = ["c_attn"]         # GPT-2 attention projection; adjust per arch

# ── Training ──────────────────────────────────────────────────────────────────
MAX_LENGTH = 256
BATCH_SIZE = 4
EPOCHS = 3
LEARNING_RATE = 3e-4

## 3. Prepare the dataset

Expected format — each line is a JSON object:
```json
{"instruction": "What caused the checkout timeout?", "response": "The timeout was caused by a slow DB query on the payments table..."}
```

In [ ]:
SAMPLE_DATA = [
    {
        "instruction": "What was the root cause of the search latency incident?",
        "response": "The root cause was a missing index on the products table, causing full table scans under high load.",
    },
    {
        "instruction": "What fixed the checkout timeout?",
        "response": "Adding a read replica and routing checkout reads there reduced primary DB load, resolving the timeout.",
    },
    {
        "instruction": "How do you mitigate database latency spikes?",
        "response": "Enable query result caching, add appropriate indexes, and use connection pooling to reduce overhead.",
    },
    {
        "instruction": "What runbook steps help during a payment service outage?",
        "response": "1. Check payment gateway status page. 2. Verify API keys are valid. 3. Roll back to last stable release if recent deploy.",
    },
    {
        "instruction": "What is retrieval-augmented generation?",
        "response": "RAG combines a retrieval system (like vector search) with a language model. The model generates answers grounded in retrieved documents.",
    },
]

# Write sample data to JSONL for reproducibility
data_path = Path("finetune_data.jsonl")
with data_path.open("w") as f:
    for row in SAMPLE_DATA:
        f.write(json.dumps(row) + "\n")

print(f"Dataset written to {data_path} ({len(SAMPLE_DATA)} examples)")

In [ ]:
raw = [json.loads(l) for l in data_path.read_text().splitlines() if l.strip()]
dataset = Dataset.from_list(raw)
dataset = dataset.train_test_split(test_size=0.2, seed=42)
print(dataset)

## 4. Load the base model and tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token   # GPT-2 has no pad token by default

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float32,
)
model.config.use_cache = False              # required for gradient checkpointing

total_params = sum(p.numel() for p in model.parameters())
print(f"Base model parameters: {total_params:,}")

## 5. Inject LoRA adapter

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

## 6. Tokenize the dataset

In [ ]:
def format_and_tokenize(batch):
    prompts = [
        f"### Instruction:\n{instr}\n\n### Response:\n{resp}{tokenizer.eos_token}"
        for instr, resp in zip(batch["instruction"], batch["response"])
    ]
    encoded = tokenizer(
        prompts,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    encoded["labels"] = encoded["input_ids"].copy()
    return encoded

tokenized = dataset.map(format_and_tokenize, batched=True, remove_columns=["instruction", "response"])
tokenized.set_format("torch")
print(tokenized)

## 7. Train

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    report_to="none",               # set to "wandb" or "mlflow" to enable tracking
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=peft_model, padding=True),
)

trainer.train()

## 8. Save the LoRA adapter

In [ ]:
peft_model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapter saved to: {OUTPUT_DIR}")

adapter_size = sum(
    f.stat().st_size for f in Path(OUTPUT_DIR).rglob("*") if f.is_file()
) / 1e6
print(f"Adapter size: {adapter_size:.1f} MB")

## 9. Load and run inference

In [ ]:
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
loaded_model = PeftModel.from_pretrained(base, OUTPUT_DIR)
loaded_model = loaded_model.merge_and_unload()  # fuse LoRA weights into base
loaded_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"Merged model saved to: {MERGED_DIR}")

In [ ]:
def generate(prompt: str, max_new_tokens: int = 150) -> str:
    inputs = tokenizer(
        f"### Instruction:\n{prompt}\n\n### Response:\n",
        return_tensors="pt"
    )
    with torch.no_grad():
        output_ids = loaded_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


response = generate("What was the root cause of the search latency incident?")
print(response)

## 10. Summary

| Parameter | Value |
|---|---|
| Base model | `distilgpt2` |
| LoRA rank (r) | 8 |
| LoRA alpha | 32 |
| Target modules | `c_attn` |
| Trainable params | ~0.3% of total |
| Epochs | 3 |
| Batch size | 4 |
| LR | 3e-4 |

**Key takeaway:** LoRA reduces trainable parameters by ~99% while adapting the model
to the target task. The adapter (`lora_adapter/`) is only a few MB and can be
swapped independently of the base model.